# Light Usage Prediction Model

This notebook builds a time series prediction model to simulate natural light usage patterns for vacation mode.

## Goal
Learn from historical light switch data and generate realistic on/off schedules that make the home appear occupied.

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
import os, pathlib, json

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Conv1D, Add, LayerNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Project modules
from features import build_features, create_sequences
from generate import generate_vacation_schedule, create_readable_schedule, export_schedule_events

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

os.makedirs('out', exist_ok=True)

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

In [ ]:
# =============================================================
# VACATION PARAMETERS — edit these before running the notebook
# =============================================================
VACATION_START = '2026-04-01'
VACATION_END   = '2026-04-08'
# =============================================================

In [ ]:
# Convert string params to Timestamps and compute duration
VACATION_START = pd.Timestamp(VACATION_START, tz='US/Pacific')
VACATION_END   = pd.Timestamp(VACATION_END,   tz='US/Pacific')
VACATION_DAYS  = (VACATION_END - VACATION_START).days
print(f'Vacation: {VACATION_START.date()} to {VACATION_END.date()} ({VACATION_DAYS} days)')

In [ ]:
# Load the data
df = pd.read_csv('out/data.csv')

# Parse timestamps as UTC, then convert to Pacific time
df['time'] = pd.to_datetime(df['time'], utc=True).dt.tz_convert('US/Pacific')

# Convert state to integer
df['state'] = df['state.value'].astype(int)

# Sort by time
df = df.sort_values('time').reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'\nDate range: {df["time"].min()} to {df["time"].max()}')
print(f'Duration: {(df["time"].max() - df["time"].min()).days} days')
print(f'\nUnique entities: {df["entity_id"].unique()}')
print(f'\nFirst few rows:')
df.head(10)

In [ ]:
# Check for missing values and data types
print("Data Info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nState value counts:")
print(df['state'].value_counts())

In [ ]:
# Create a pivot table to see all lights at once
# First, let's create a continuous time series

# Get the full time range
start_time = df['time'].min().floor('15min')
end_time = df['time'].max().ceil('15min')

# Create a complete time index at 15-minute intervals
time_index = pd.date_range(start=start_time, end=end_time, freq='15min', tz='US/Pacific')

# Get unique entities
entities = list(df['entity_id'].unique())
print(f"Entities: {entities}")

# Create a dataframe for each entity and merge
all_lights = pd.DataFrame(index=time_index)

for entity in entities:
    entity_df = df[df['entity_id'] == entity][['time', 'state']].copy()
    entity_df = entity_df.set_index('time')

    # Resample to 15-minute intervals, forward-fill to maintain state
    entity_resampled = entity_df.resample('15min').last().ffill().fillna(0)

    entity_resampled.columns = [entity]
    all_lights = all_lights.join(entity_resampled)

# Fill any remaining NaNs with 0 (lights start off)
all_lights = all_lights.fillna(0).astype(int)

print(f"\nContinuous time series shape: {all_lights.shape}")
print(f"Time range: {all_lights.index.min()} to {all_lights.index.max()}")
all_lights.head()

In [ ]:
# Visualize light usage over time (Gantt-style)
fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)

for idx, entity in enumerate(entities):
    ax = axes[idx]
    
    # Plot as filled area
    ax.fill_between(all_lights.index, 0, all_lights[entity], 
                     alpha=0.7, label=entity)
    
    ax.set_ylabel('State', fontsize=10)
    ax.set_ylim(-0.1, 1.1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Off', 'On'])
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date', fontsize=12)
plt.suptitle('Light Usage Patterns Over Time', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Add time-based features for analysis
all_lights_analysis = all_lights.copy()
all_lights_analysis['hour'] = all_lights_analysis.index.hour
all_lights_analysis['day_of_week'] = all_lights_analysis.index.dayofweek
all_lights_analysis['is_weekend'] = all_lights_analysis['day_of_week'].isin([5, 6]).astype(int)

# Analyze usage by hour of day
hourly_usage = all_lights_analysis.groupby('hour')[entities].mean()

fig, ax = plt.subplots(figsize=(12, 5))
hourly_usage.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Probability Light is On', fontsize=12)
ax.set_title('Average Light Usage by Hour of Day', fontsize=14)
ax.legend(title='Light')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nPeak usage hours for each light:")
for entity in entities:
    peak_hour = hourly_usage[entity].idxmax()
    peak_prob = hourly_usage[entity].max()
    print(f"{entity}: Hour {peak_hour} ({peak_prob:.2%} on-time)")

In [ ]:
# Analyze weekday vs weekend patterns
weekday_usage = all_lights_analysis[all_lights_analysis['is_weekend'] == 0][entities].mean()
weekend_usage = all_lights_analysis[all_lights_analysis['is_weekend'] == 1][entities].mean()

comparison = pd.DataFrame({
    'Weekday': weekday_usage,
    'Weekend': weekend_usage
})

print("Weekday vs Weekend Usage:")
print(comparison)

comparison.plot(kind='bar', figsize=(10, 5))
plt.title('Light Usage: Weekday vs Weekend', fontsize=14)
plt.ylabel('Probability Light is On', fontsize=12)
plt.xlabel('Light', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Day Type')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze simultaneous light usage (correlation)
print("Light State Correlation Matrix:")
correlation = all_lights[entities].corr()
print(correlation)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Light State Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print("\nSimultaneous usage statistics:")
print(f"All lights on together: {(all_lights[entities].sum(axis=1) == 3).sum()} times ({(all_lights[entities].sum(axis=1) == 3).mean():.2%})")
print(f"All lights off together: {(all_lights[entities].sum(axis=1) == 0).sum()} times ({(all_lights[entities].sum(axis=1) == 0).mean():.2%})")

In [ ]:
# Calculate daily total on-time for each light
daily_on_time = all_lights.resample('D')[entities].sum() * 0.25  # 15-min intervals * 0.25 = hours

fig, ax = plt.subplots(figsize=(14, 5))
daily_on_time.plot(ax=ax, marker='o')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Hours On', fontsize=12)
ax.set_title('Daily On-Time per Light', fontsize=14)
ax.legend(title='Light')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nDaily on-time statistics (hours):")
print(daily_on_time.describe())

## 4. Feature Engineering

In [ ]:
features_df, input_features, target_features = build_features(all_lights, entities)

print(f"Feature dataframe shape: {features_df.shape}")
print(f"\nInput features ({len(input_features)}): {input_features}")
print(f"Target features ({len(target_features)}): {target_features}")
features_df.head()

## 5. Data Preprocessing

In [ ]:
SEQUENCE_LENGTH = 96  # Look back 96 time steps (24 hours with 15-min intervals)
PREDICTION_HORIZON = 1  # Predict next single time step

print(f"Sequence length: {SEQUENCE_LENGTH} steps ({SEQUENCE_LENGTH * 15} minutes = {SEQUENCE_LENGTH * 15 // 60} hours)")

print(f"\nOn-rate per light:")
for tf in target_features:
    rate = features_df[tf].mean()
    print(f"  {tf}: {rate:.1%} of steps")

In [ ]:
X, y = create_sequences(
    features_df,
    input_features,
    target_features,
    SEQUENCE_LENGTH,
    PREDICTION_HORIZON
)

print(f"X shape: {X.shape} (samples, sequence_length, features)")
print(f"y shape: {y.shape} (samples, targets)")
print(f"\nTotal sequences created: {len(X)}")

In [ ]:
# Train/validation/test split (temporal split)
# Use last 15% for test, 15% before that for validation
train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size+val_size]
y_val = y[train_size:train_size+val_size]

X_test = X[train_size+val_size:]
y_test = y[train_size+val_size:]

print(f"Training set: {X_train.shape[0]} sequences")
print(f"Validation set: {X_val.shape[0]} sequences")
print(f"Test set: {X_test.shape[0]} sequences")
print(f"\nClass distribution in training set:")
for i, entity in enumerate(target_features):
    on_pct = y_train[:, i].mean()
    print(f"  {entity}: {on_pct:.2%} on")

## 6. Model Definition

In [ ]:
def tcn_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    """Single TCN residual block: two causal convolutions with a skip connection."""
    conv = Conv1D(filters, kernel_size, padding='causal',
                  dilation_rate=dilation_rate, activation='relu')(x)
    conv = Dropout(dropout_rate)(conv)
    conv = Conv1D(filters, kernel_size, padding='causal',
                  dilation_rate=dilation_rate, activation='relu')(conv)
    conv = Dropout(dropout_rate)(conv)

    # Match dimensions for residual connection if needed
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1, padding='same')(x)

    out = Add()([x, conv])
    out = LayerNormalization()(out)
    return out


def build_model(sequence_length, n_features, n_outputs, filters=64, kernel_size=3,
                dropout_rate=0.2):
    """
    Build a TCN (Temporal Convolutional Network) for light state prediction.

    Uses dilated causal convolutions with exponentially increasing dilation rates
    so deeper layers see further back in time. With kernel_size=3 and dilations
    [1, 2, 4, 8, 16, 32], the receptive field is 127 steps — covering the full
    96-step (24-hour) input window.
    """
    inp = Input(shape=(sequence_length, n_features))

    x = inp
    for dilation in [1, 2, 4, 8, 16, 32]:
        x = tcn_block(x, filters, kernel_size, dilation, dropout_rate)

    # Take the last timestep's output (causal — only looks backward)
    x = x[:, -1, :]

    x = Dense(32, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(n_outputs, activation='sigmoid')(x)

    model = Model(inputs=inp, outputs=x)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


model = build_model(
    sequence_length=SEQUENCE_LENGTH,
    n_features=len(input_features),
    n_outputs=len(target_features),
    filters=64,
    kernel_size=3,
    dropout_rate=0.2
)

print("Model architecture:")
model.summary()

## 7. Model Training

In [ ]:
# Define callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=0.00001,
        verbose=1
    ),
    ModelCheckpoint(
        'out/best_light_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Train the model
print("Training model...\n")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Model Loss', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Model Accuracy', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Make predictions
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

# Per-light accuracy
print("\nPer-light accuracy on test set:")
for i, entity in enumerate(target_features):
    accuracy = (y_pred[:, i] == y_test[:, i]).mean()
    print(f"  {entity}: {accuracy:.2%}")

In [ ]:
# Confusion matrices for each light
from sklearn.metrics import confusion_matrix, classification_report

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, entity in enumerate(target_features):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Off', 'On'], yticklabels=['Off', 'On'])
    axes[i].set_title(f'{entity}', fontsize=12)
    axes[i].set_ylabel('Actual', fontsize=10)
    axes[i].set_xlabel('Predicted', fontsize=10)

plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Detailed classification reports
print("\nDetailed Classification Reports:\n")
for i, entity in enumerate(target_features):
    print(f"\n{entity}:")
    print(classification_report(y_test[:, i], y_pred[:, i],
                                target_names=['Off', 'On']))

In [ ]:
# Visualize predictions vs actual for a sample period
# Take a 3-day window from test set
sample_days = 3
samples_per_day = 96  # 24 hours * 4 (15-min intervals)
sample_size = sample_days * samples_per_day

if len(y_test) >= sample_size:
    sample_start = len(y_test) - sample_size

    fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)

    for i, entity in enumerate(target_features):
        ax = axes[i]
        x = range(sample_size)

        ax.fill_between(x, 0, y_test[sample_start:, i],
                        alpha=0.5, label='Actual', color='blue')
        ax.plot(x, y_pred[sample_start:, i] - 0.1,
               label='Predicted', color='red', linewidth=2, alpha=0.7)

        ax.set_ylabel(entity, fontsize=10)
        ax.set_ylim(-0.3, 1.2)
        ax.set_yticks([0, 1])
        ax.set_yticklabels(['Off', 'On'])
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time Steps (15-min intervals)', fontsize=12)
    plt.suptitle(f'Predictions vs Actual (Last {sample_days} Days of Test Set)',
                fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()
else:
    print(f"Not enough test data for {sample_days}-day visualization")

In [ ]:
print(f"Generating {VACATION_DAYS}-day vacation schedule: {VACATION_START.date()} to {VACATION_END.date()}...\n")

vacation_schedule = generate_vacation_schedule(
    model=model,
    start_date=VACATION_START,
    num_days=VACATION_DAYS,
    sequence_length=SEQUENCE_LENGTH,
    input_features=input_features,
    entities=entities,
    features_df=features_df,
    temperature=1.0
)

print(f"Schedule generated: {vacation_schedule.shape[0]} time steps")
print(f"\nFirst few entries:")
print(vacation_schedule.head(20))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)

for i, entity in enumerate(entities):
    ax = axes[i]
    ax.fill_between(vacation_schedule.index, 0, vacation_schedule[entity], alpha=0.7, label=entity)
    ax.set_ylabel('State', fontsize=10)
    ax.set_ylim(-0.1, 1.1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Off', 'On'])
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date & Time', fontsize=12)
plt.suptitle(f'Generated Vacation Schedule ({VACATION_START.date()} \u2013 {VACATION_END.date()})', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Compare generated schedule statistics to actual data
print("Comparison of Generated vs Actual Light Usage:\n")

# Daily on-time
vacation_daily_on = vacation_schedule.resample('D').sum() * 0.25  # hours
actual_daily_on = all_lights[entities].resample('D').sum() * 0.25

comparison_stats = pd.DataFrame({
    'Actual_Mean': actual_daily_on.mean(),
    'Actual_Std': actual_daily_on.std(),
    'Generated_Mean': vacation_daily_on.mean(),
    'Generated_Std': vacation_daily_on.std()
})

print("Daily On-Time (hours):")
print(comparison_stats)

# Hourly usage patterns
vacation_hourly = vacation_schedule.copy()
vacation_hourly['hour'] = vacation_hourly.index.hour
vacation_hourly_mean = vacation_hourly.groupby('hour')[entities].mean()

actual_hourly_mean = all_lights_analysis.groupby('hour')[entities].mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, entity in enumerate(entities):
    ax = axes[i]

    x = range(24)
    ax.plot(x, actual_hourly_mean[entity], marker='o', label='Actual', linewidth=2)
    ax.plot(x, vacation_hourly_mean[entity], marker='s', label='Generated', linewidth=2)

    ax.set_xlabel('Hour of Day', fontsize=10)
    ax.set_ylabel('Probability On', fontsize=10)
    ax.set_title(entity, fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, 24, 3))

plt.suptitle('Hourly Usage Pattern: Generated vs Actual', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Show readable schedule for each light
for entity in entities:
    print(f"\n{'='*80}")
    print(f"{entity.upper()} - ON PERIODS")
    print(f"{'='*80}")

    readable = create_readable_schedule(vacation_schedule, entity)
    on_periods = readable[readable['state'] == 'ON']

    if len(on_periods) > 0:
        for _, period in on_periods.iterrows():
            print(f"{period['start'].strftime('%Y-%m-%d %H:%M')} -> "
                  f"{period['end'].strftime('%H:%M')} "
                  f"({period['duration']:.1f} hours)")
    else:
        print("No ON periods in this schedule")

In [ ]:
all_events = export_schedule_events(
    vacation_schedule, entities,
    entity_map_path='out/entity_map.json',
    output_path='out/schedule_events.json'
)

print(f'Exported {len(all_events)} events to out/schedule_events.json\n')
for e in all_events:
    print(f"  {e['time']}  {e['action']:8}  {e['entity_id']}")